# Comprehensive Unsupervised Learning Masterclass

Unlike Supervised Learning, Unsupervised Learning algorithms deal with unlabeled data. Their goal is to find hidden structures, patterns, or groupings within the data.

### Table of Contents
1. [Data Loading (Wine Dataset)](#1.-Data-Loading-(Wine-Dataset))
2. [Dimensionality Reduction: Principal Component Analysis (PCA)](#2.-Dimensionality-Reduction:-Principal-Component-Analysis-(PCA))
3. [Clustering: K-Means Algorithm](#3.-Clustering:-K-Means-Algorithm)
\n

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
\n

## 1. Data Loading (Wine Dataset)
We will use the Wine dataset, which has 13 features describing chemical properties of wines. Even though we have labels (3 classes of wine), we will **hide the labels** from our algorithms to simulate unsupervised learning.\n

In [ ]:
# Load dataset
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y_true = data.target # We keep this only to evaluate how well our unsupervised methods did later!

# Unsupervised models are very sensitive to scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset shape: {X.shape}")
\n

## 2. Dimensionality Reduction: Principal Component Analysis (PCA)

### Theory & Intuition
Visualizing 13 dimensions is impossible for humans. PCA is a mathematical procedure that transforms a number of (possibly) correlated variables into a (smaller) number of uncorrelated variables called **Principal Components**.

**How it works:**
1. Compute the Covariance Matrix of the features.
2. Calculate the Eigenvectors and Eigenvalues of this matrix.
3. Sort the Eigenvectors by descending Eigenvalues. The Eigenvectors are the new "axes" (Principal Components), and the Eigenvalues tell us how much variance (information) that component holds.

Let's reduce our 13 features down to just 2 dimensions so we can plot it!\n

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Variance explained by PC1: {pca.explained_variance_ratio_[0]:.2f}")
print(f"Variance explained by PC2: {pca.explained_variance_ratio_[1]:.2f}")
print(f"Total variance retained: {sum(pca.explained_variance_ratio_):.2f}")

# Plotting the PCA
plt.figure(figsize=(8, 6))
# We color by the true labels just to see if PCA naturally separated the classes
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='viridis', alpha=0.8, edgecolor='k')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Wine Dataset (2D Projection)')
plt.colorbar(scatter, label='True Wine Class')
plt.show()
\n

## 3. Clustering: K-Means Algorithm

### Theory & Intuition
K-Means aims to partition `n` observations into `k` clusters in which each observation belongs to the cluster with the nearest mean (centroid).

**The Algorithm:**
1. Randomly initialize `K` centroids.
2. **Assignment Step**: Assign each data point to the nearest centroid (using Euclidean distance).
3. **Update Step**: Move the centroid to the mean (average) position of all points assigned to it.
4. Repeat Steps 2 and 3 until the centroids stop moving (convergence).

Let's apply K-Means to our 2D PCA data to see if it can find the 3 clusters automatically.\n

In [ ]:
# We ask K-Means to find 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X_pca)

# Evaluate the clustering quality without knowing the true labels
# Silhouette score measures how similar an object is to its own cluster compared to other clusters.
sil_score = silhouette_score(X_pca, cluster_labels)
print(f"Silhouette Score: {sil_score:.3f} (closer to 1 is better)")

# Plot the K-Means results
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='plasma', alpha=0.8, edgecolor='k')

# Plot the centroids
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1], s=200, c='red', marker='X', label='Centroids')

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('K-Means Clustering on PCA-reduced Data')
plt.legend()
plt.show()
\n